# Week 6, Day 1 — Welcome to the Model Context Protocol
### Local Models Edition — by Abhishek

The epic finale week — and we're back on the **OpenAI Agents SDK**, this
time pointed entirely at a **free local model** through Ollama's
OpenAI-compatible endpoint. No `OPENAI_API_KEY`, no spend.

MCP lets an agent use tools someone else wrote, running as a separate
program. A small client spawns the server as a subprocess; the server
reports back the tools it offers. Once you can list one server's tools, you
can use any of them the same way — and this part of the course was already
100% free and local in the original, so we keep it as-is.


## This week needs a local server, not Colab

Weeks 1-5 mostly worked on Colab too. This week launches local subprocess
MCP servers (via `npx`/`uvx`) and, later, a local trading floor — it needs
**your own PC with Ollama running.** If you're on Colab, treat this week as
a reading/local-follow-along week.


## 0. Setup

In [ ]:
import subprocess
node_v = subprocess.run(["node", "--version"], capture_output=True, text=True).stdout
npx_v = subprocess.run(["npx", "--version"], capture_output=True, text=True).stdout
print("node:", node_v or "NOT FOUND - install from nodejs.org")
print("npx:", npx_v or "NOT FOUND")


In [ ]:
%pip install -q openai-agents mcp python-dotenv


## Pointing the OpenAI Agents SDK at a local model

Ollama exposes an OpenAI-compatible endpoint on `http://localhost:11434/v1`.
We set this once, and every `Agent` we build for the rest of this week runs
on your local model with zero API spend.


In [ ]:
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, set_default_openai_client, set_tracing_disabled
from agents.mcp import MCPServerStdio
from IPython.display import Image, display

# Point the SDK at your local Ollama server instead of OpenAI
local_client = AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
set_default_openai_client(local_client)
set_tracing_disabled(True)  # tracing normally posts to platform.openai.com; disable for local runs

MODEL_NAME = "llama3.2:3b"
print("Agents SDK configured for local model:", MODEL_NAME)


### A quick note for Windows

Launching a local MCP server from a notebook hits one rough edge on
Windows: the server writes startup output to stderr, but inside a Windows
Jupyter kernel that stream has no real file handle, so the launch fails
with `io.UnsupportedOperation: fileno`. Mac and Linux are unaffected. The
fix: send the server's stderr to the null device.


In [ ]:
import sys
if sys.platform == "win32":
    import functools
    import subprocess
    import mcp.client.stdio as mcp_stdio
    mcp_stdio.stdio_client = functools.partial(mcp_stdio.stdio_client, errlog=subprocess.DEVNULL)
    print("Applied the Windows adjustment")
else:
    print("Not Windows, nothing to do here")


## Server 1: Fetch

A small MCP server that fetches and cleans up a web page, wrapping it as a
single `fetch` tool — launched with `uvx` (Python's `npx` equivalent, pulls
from PyPI on demand, no separate install step).


In [ ]:
fetch_params = {"command": "uvx", "args": ["mcp-server-fetch"]}

async with MCPServerStdio(params=fetch_params, client_session_timeout_seconds=30) as server:
    fetch_tools = await server.list_tools()

for t in fetch_tools:
    print(t.name, "-", t.description[:80])


## Server 2: a sandboxed filesystem

Exposes read/write/list tools scoped to a single folder you choose — the
agent can never touch anything outside it.


In [ ]:
import os
sandbox = os.path.abspath("sandbox")
os.makedirs(sandbox, exist_ok=True)

files_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", sandbox]}

async with MCPServerStdio(params=files_params, client_session_timeout_seconds=30) as server:
    file_tools = await server.list_tools()

for t in file_tools:
    print(t.name, "-", t.description[:80])


## Server 3: memory

A small local knowledge-graph server — entities, relations, observations —
persisted to a JSON file so an agent can remember things across runs.


In [ ]:
memory_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-memory"]}

async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=30) as server:
    memory_tools = await server.list_tools()

for t in memory_tools:
    print(t.name, "-", t.description[:80])


## Handing servers to an agent

An `Agent` just takes a list of `mcp_servers` alongside its model and
instructions — everything after this is the same whether the model is
GPT-4-class or a free 3B local model.


In [ ]:
async with MCPServerStdio(params=fetch_params, client_session_timeout_seconds=30) as fetch_server, \
           MCPServerStdio(params=files_params, client_session_timeout_seconds=30) as files_server:

    agent = Agent(
        name="researcher",
        instructions="You research topics using your fetch tool and save findings to files using your filesystem tools.",
        model=MODEL_NAME,
        mcp_servers=[fetch_server, files_server],
    )

    result = await Runner.run(agent, "Fetch https://news.ycombinator.com and save the page title to a file called title.txt")
    print(result.final_output)


In [ ]:
with open(os.path.join(sandbox, "title.txt")) as f:
    print(f.read())


## Recap, and where we are heading

You pointed the OpenAI Agents SDK at a completely free local model, met
three ready-made MCP servers (fetch, filesystem, memory), and ran an agent
against two of them together.

Tomorrow: building your own MCP server — the accounts system for this
week's trading floor project.

## Exercise
Give the agent all three servers at once, and ask it to research a topic,
remember a fact about it in the memory server, then save a summary to a
file. Confirm the memory persisted by starting a fresh agent run and asking
it what it remembers.
